In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import files

COLORS = {
    "Red": [
        (np.array([0, 120, 70]),   np.array([10, 255, 255]),  (0, 0, 255)),
        (np.array([170, 120, 70]), np.array([180, 255, 255]), (0, 0, 255)),
    ],
    "Green":  [(np.array([36, 100, 100]), np.array([86, 255, 255]),  (0, 200, 0))],
    "Blue":   [(np.array([100, 150, 50]), np.array([130, 255, 255]), (255, 100, 0))],
    "Yellow": [(np.array([20, 100, 100]), np.array([35, 255, 255]),  (0, 220, 220))],
}

MIN_AREA = 2000

print("Upload an image")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
frame = cv2.imread(image_path)
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

for color_name, ranges in COLORS.items():
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    bgr = ranges[0][2]
    for (lo, hi, _) in ranges:
        mask = cv2.bitwise_or(mask, cv2.inRange(hsv, lo, hi))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(frame, (x, y), (x + w, y + h), bgr, 2)
        cv2.putText(frame, color_name, (x + 3, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

cv2_imshow(frame)